# 04 — Interactive PEFT Explorer

A lightweight playground for attendees to **choose a PEFT method**, tweak a few constraints, and immediately see:

- **where the method operates**
- **how many parameters it trains**
- **what it tends to be good / bad at**
- a **quick toy training run** with curves and a method-specific visualization

This notebook is intentionally small and opinionated. It is meant to support discussion around **“what should I use when?”**, not to be a definitive benchmark.

## Setup

Run the next cell once. In Google Colab it installs the widget dependency and enables interactive controls.


In [ ]:
# --- Setup (run this first) ---
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip -q install ipywidgets matplotlib torch torchvision
    from google.colab import output
    output.enable_custom_widget_manager()

import ipywidgets as widgets
widgets.Dropdown(options=["linear_probing", "lora", "adapters"], description="Widget check:")


In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display, clear_output
import matplotlib.pyplot as plt
import ipywidgets as widgets

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def set_seed(seed: int = 7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(7)


## 1) Method cards

We keep a small set of methods that cover distinct intervention styles:

- **linear probing** — readout only
- **prompt tuning** — input / conditioning space
- **adapters** — hidden-state computation
- **LoRA** — low-rank update in weight space
- **BitFit** — biases only
- **partial finetuning** — a small subset of backbone weights


In [ ]:
PEFT_CARDS: Dict[str, Dict] = {
    "linear_probing": {
        "family": "baseline",
        "where": "readout head only",
        "updates": "classifier weights",
        "good_for": ["very cheap baseline", "feature quality check", "small data"],
        "watch_out": ["plateaus early", "cannot reshape backbone features"],
        "best_when": "pretrained features already separate the task reasonably well",
        "visualization": "classifier weight heatmap",
    },
    "prompt_tuning": {
        "family": "prompt / input-space",
        "where": "learned prompt tokens prepended to the input sequence",
        "updates": "soft prompt embeddings only",
        "good_for": ["tiny adaptation footprint", "swap task-specific prompts easily", "preserve backbone strictly"],
        "watch_out": ["can be optimization-sensitive", "may lag when task needs deeper change"],
        "best_when": "you want tiny, easily swappable task-specific state and a frozen backbone",
        "visualization": "learned prompt token norms",
    },
    "adapters": {
        "family": "bottleneck adapters",
        "where": "inside the network as small trainable residual modules",
        "updates": "down/up projection adapter weights",
        "good_for": ["modularity", "explicit hidden-state intervention", "multi-task packaging"],
        "watch_out": ["extra latency / modules", "more moving parts than a linear probe"],
        "best_when": "you want per-task modules that modify hidden computation directly",
        "visualization": "adapter activation distribution",
    },
    "lora": {
        "family": "LoRA / low-rank",
        "where": "weight space via low-rank update on selected linear layers",
        "updates": "A and B low-rank factors only",
        "good_for": ["strong default PEFT baseline", "good quality/efficiency tradeoff", "widespread ecosystem support"],
        "watch_out": ["rank choice matters", "still layer/target selection decisions"],
        "best_when": "you want a practical first PEFT baseline with strong quality per parameter",
        "visualization": "effective low-rank weight delta heatmap",
    },
    "bitfit": {
        "family": "bias-only",
        "where": "existing bias terms in the network",
        "updates": "bias vectors only",
        "good_for": ["extremely parameter-efficient", "simple to explain", "cheap sanity check"],
        "watch_out": ["limited expressive power", "often weaker than LoRA/adapters"],
        "best_when": "you need an ultra-cheap adaptation sanity check",
        "visualization": "bias change histogram",
    },
    "partial_finetuning": {
        "family": "partial FT",
        "where": "selected backbone blocks",
        "updates": "subset of backbone weights",
        "good_for": ["stronger adaptation than tiny PEFT", "simple mental model", "bridges PEFT to finetuning"],
        "watch_out": ["more memory and optimizer state", "less modular than PEFT"],
        "best_when": "you can afford a bit more training budget and need deeper adaptation",
        "visualization": "updated last-layer weight norms",
    },
}
list(PEFT_CARDS.keys())


## 2) Tiny synthetic sequence task

To keep runtime low, we use a frozen sequence backbone and a small synthetic classification task.

The task regime changes **how much the target decision differs from the pretrained representation**:

- **easy / aligned**: features are already pretty good
- **medium shift**: some adaptation helps
- **hard / shifted**: deeper interventions benefit more

This is not supposed to mimic every real task. It is just enough structure to make the methods behave differently.


In [ ]:
def make_sequence_dataset(n=512, seq_len=8, vocab_size=16, regime="easy", seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.randint(low=0, high=vocab_size, size=(n, seq_len), generator=g)

    # Base features extracted directly from token ids.
    s_first = x[:, : seq_len // 2].float().sum(dim=1)
    s_second = x[:, seq_len // 2 :].float().sum(dim=1)
    parity = (x % 2 == 0).float().sum(dim=1)
    max_tok = x.float().max(dim=1).values
    center = x[:, 2:6].float().mean(dim=1)

    if regime == "easy":
        score = 0.8 * (s_first - s_second) + 0.2 * parity
    elif regime == "medium":
        score = 0.4 * (s_first - s_second) + 0.4 * (center - 7.0) + 0.4 * (max_tok > 11).float()
    elif regime == "hard":
        score = 0.5 * torch.sin(center) + 0.7 * ((max_tok > 12).float()) - 0.35 * (parity - 4.0).abs()
    else:
        raise ValueError(f"Unknown regime: {regime}")

    y = (score > score.median()).long()
    return x, y

def split_dataset(x, y, train_frac=0.7):
    n = len(x)
    n_train = int(train_frac * n)
    return (x[:n_train], y[:n_train]), (x[n_train:], y[n_train:])


## 3) Small frozen backbone and PEFT wrappers

In [ ]:

class TinyBackbone(nn.Module):
    def __init__(self, vocab_size=16, d_model=32):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.proj1 = nn.Linear(d_model, d_model)
        self.proj2 = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

        # Build a mildly structured checkpoint by hand:
        with torch.no_grad():
            nn.init.normal_(self.embed.weight, std=0.5)
            nn.init.xavier_uniform_(self.proj1.weight)
            nn.init.zeros_(self.proj1.bias)
            nn.init.xavier_uniform_(self.proj2.weight)
            nn.init.zeros_(self.proj2.bias)

    def encode(self, x_emb):
        h = self.proj1(x_emb)
        h = torch.tanh(h)
        h = self.proj2(h)
        h = self.norm(h)
        pooled = h.mean(dim=1)
        return pooled, h

    def forward(self, token_ids):
        x = self.embed(token_ids)
        return self.encode(x)

class LoRALinear(nn.Module):
    def __init__(self, linear: nn.Linear, rank=4, alpha=1.0):
        super().__init__()
        self.linear = linear
        self.rank = rank
        self.alpha = alpha
        in_features = linear.in_features
        out_features = linear.out_features

        self.A = nn.Parameter(torch.randn(in_features, rank) * 0.02)
        self.B = nn.Parameter(torch.zeros(rank, out_features))
        for p in self.linear.parameters():
            p.requires_grad = False

    def forward(self, x):
        base = self.linear(x)
        delta = x @ self.A @ self.B * (self.alpha / self.rank)
        return base + delta

    def delta_weight(self):
        return (self.A @ self.B).detach().cpu()

class AdapterBlock(nn.Module):
    def __init__(self, d_model=32, bottleneck=8):
        super().__init__()
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)

    def forward(self, h):
        return h + self.up(torch.relu(self.down(h)))

class PEFTClassifier(nn.Module):
    def __init__(self, method="linear_probing", vocab_size=16, d_model=32, prompt_len=4, rank=4, bottleneck=8):
        super().__init__()
        self.method = method
        self.backbone = TinyBackbone(vocab_size=vocab_size, d_model=d_model)
        self.prompt_len = prompt_len
        self.adapter = None
        self.prompt = None

        # Freeze everything by default, then selectively unfreeze / insert modules.
        for p in self.backbone.parameters():
            p.requires_grad = False

        if method == "lora":
            self.backbone.proj1 = LoRALinear(self.backbone.proj1, rank=rank)
            self.backbone.proj2 = LoRALinear(self.backbone.proj2, rank=rank)
        elif method == "adapters":
            self.adapter = AdapterBlock(d_model=d_model, bottleneck=bottleneck)
        elif method == "prompt_tuning":
            self.prompt = nn.Parameter(torch.randn(prompt_len, d_model) * 0.02)
        elif method == "bitfit":
            for name, p in self.backbone.named_parameters():
                if "bias" in name:
                    p.requires_grad = True
        elif method == "partial_finetuning":
            for name, p in self.backbone.named_parameters():
                if name.startswith("proj2") or name.startswith("norm"):
                    p.requires_grad = True
        elif method == "linear_probing":
            pass
        else:
            raise ValueError(f"Unsupported method: {method}")

        self.head = nn.Linear(d_model, 2)

    def forward(self, token_ids):
        x_emb = self.backbone.embed(token_ids)

        if self.method == "prompt_tuning":
            batch = x_emb.shape[0]
            p = self.prompt.unsqueeze(0).expand(batch, -1, -1)
            x_emb = torch.cat([p, x_emb], dim=1)

        pooled, h = self.backbone.encode(x_emb)

        if self.method == "adapters":
            h = self.adapter(h)
            pooled = h.mean(dim=1)

        logits = self.head(pooled)
        return logits, {"pooled": pooled, "hidden": h}

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_quick(method="lora", regime="easy", train_samples=256, epochs=15, lr=3e-3, seed=7):
    set_seed(seed)
    x, y = make_sequence_dataset(n=max(512, train_samples * 2), regime=regime, seed=seed)
    (x_train, y_train), (x_val, y_val) = split_dataset(x, y)
    x_train, y_train = x_train[:train_samples], y_train[:train_samples]

    model = PEFTClassifier(method=method).to(DEVICE)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)

    train_loss_hist, val_acc_hist = [], []

    x_train = x_train.to(DEVICE)
    y_train = y_train.to(DEVICE)
    x_val = x_val.to(DEVICE)
    y_val = y_val.to(DEVICE)

    for _ in range(epochs):
        model.train()
        logits, aux = model(x_train)
        loss = F.cross_entropy(logits, y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            val_logits, val_aux = model(x_val)
            val_acc = (val_logits.argmax(dim=1) == y_val).float().mean().item()

        train_loss_hist.append(loss.item())
        val_acc_hist.append(val_acc)

    result = {
        "model": model,
        "train_loss": train_loss_hist,
        "val_acc": val_acc_hist,
        "trainable_params": count_trainable_params(model),
        "aux": val_aux,
        "x_val": x_val,
        "y_val": y_val,
    }
    return result


def score_run(best_acc, trainable_params, budget_params):
    efficiency_penalty = 0.02 * math.log10(max(trainable_params, 1))
    over_budget_penalty = 0.0
    if budget_params is not None and trainable_params > budget_params:
        over_budget_penalty = 0.15 + 0.15 * ((trainable_params - budget_params) / max(budget_params, 1))
    return best_acc - efficiency_penalty - over_budget_penalty, over_budget_penalty

def recommendation_text(method, regime, trainable_params):
    card = PEFT_CARDS[method]
    lines = [
        f"**Family:** {card['family']}",
        f"**Operates in:** {card['where']}",
        f"**Updates:** {card['updates']}",
        f"**Trainable params in this toy setup:** {trainable_params:,}",
        "",
        f"**Usually a good fit when:** {card['best_when']}.",
    ]

    if regime == "easy":
        lines.append("**In an easy/aligned regime:** cheap methods often do surprisingly well; check whether a linear probe already solves enough of the problem.")
    elif regime == "medium":
        lines.append("**In a medium-shift regime:** methods that can change internal computation a bit more (LoRA / adapters / partial FT) often become more attractive.")
    elif regime == "hard":
        lines.append("**In a hard/shifted regime:** very tiny methods may struggle; deeper intervention often matters more than absolute parameter frugality.")

    lines.append("")
    lines.append("**Strengths:** " + ", ".join(card["good_for"]))
    lines.append("**Watch-outs:** " + ", ".join(card["watch_out"]))
    return "\n".join(lines)

def method_specific_plot(result, method):
    model = result["model"]
    if method == "lora":
        fig, ax = plt.subplots(figsize=(5, 4))
        delta = model.backbone.proj1.delta_weight().numpy()
        im = ax.imshow(delta, aspect="auto")
        ax.set_title("LoRA effective ΔW (proj1)")
        ax.set_xlabel("output dim")
        ax.set_ylabel("input dim")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.show()

    elif method == "prompt_tuning":
        fig, ax = plt.subplots(figsize=(5, 3))
        norms = model.prompt.detach().cpu().norm(dim=1).numpy()
        ax.bar(np.arange(len(norms)), norms)
        ax.set_title("Learned prompt token norms")
        ax.set_xlabel("prompt token index")
        ax.set_ylabel("L2 norm")
        plt.show()

    elif method == "adapters":
        with torch.no_grad():
            hidden = result["aux"]["hidden"].detach().cpu()
            act = model.adapter.down(hidden).reshape(-1).numpy()
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.hist(act, bins=30)
        ax.set_title("Adapter bottleneck activations")
        ax.set_xlabel("activation")
        ax.set_ylabel("count")
        plt.show()

    elif method == "bitfit":
        biases = []
        for n, p in model.backbone.named_parameters():
            if "bias" in n and p.requires_grad:
                biases.append(p.detach().cpu().reshape(-1))
        all_bias = torch.cat(biases).numpy()
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.hist(all_bias, bins=30)
        ax.set_title("Trainable bias values after adaptation")
        ax.set_xlabel("bias value")
        ax.set_ylabel("count")
        plt.show()

    elif method == "linear_probing":
        fig, ax = plt.subplots(figsize=(5, 3))
        w = model.head.weight.detach().cpu().numpy()
        im = ax.imshow(w, aspect="auto")
        ax.set_title("Linear head weights")
        ax.set_xlabel("feature dim")
        ax.set_ylabel("class")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.show()

    elif method == "partial_finetuning":
        w = model.backbone.proj2.weight.detach().cpu()
        norms = w.norm(dim=1).numpy()
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.plot(norms)
        ax.set_title("Last block row norms")
        ax.set_xlabel("row index")
        ax.set_ylabel("L2 norm")
        plt.show()

def run_and_render(method, regime, train_samples, epochs, lr, seed, budget_params=None):
    result = train_quick(
        method=method,
        regime=regime,
        train_samples=train_samples,
        epochs=epochs,
        lr=lr,
        seed=seed,
    )

    best_acc = max(result["val_acc"])
    final_acc = result["val_acc"][-1]
    score, over_budget_penalty = score_run(best_acc, result["trainable_params"], budget_params)

    display(Markdown(f"## {method}"))
    display(Markdown(recommendation_text(method, regime, result["trainable_params"])))

    if budget_params is not None:
        budget_msg = (
            f"**Budget check:** `{result['trainable_params']:,}` trainable params "
            f"vs budget `{budget_params:,}`."
        )
        if result["trainable_params"] > budget_params:
            budget_msg += " This run is **over budget**, so the score is penalized."
        else:
            budget_msg += " This run is **within budget**."
        display(Markdown(budget_msg))

    fig, ax = plt.subplots(figsize=(5, 3))
    ax.plot(result["train_loss"])
    ax.set_title("Training loss")
    ax.set_xlabel("epoch")
    ax.set_ylabel("cross-entropy")
    plt.show()

    fig, ax = plt.subplots(figsize=(5, 3))
    ax.plot(result["val_acc"])
    ax.set_title("Validation accuracy")
    ax.set_xlabel("epoch")
    ax.set_ylabel("accuracy")
    plt.show()

    method_specific_plot(result, method)

    leaderboard_block = "\n".join([
        "### Leaderboard-ready result",
        f"- Method: `{method}`",
        f"- Regime: `{regime}`",
        f"- Train samples: `{train_samples}`",
        f"- Epochs: `{epochs}`",
        f"- Learning rate: `{lr:.1e}`",
        f"- Seed: `{seed}`",
        f"- Trainable params: `{result['trainable_params']:,}`",
        f"- Best val accuracy: `{best_acc:.3f}`",
        f"- Final val accuracy: `{final_acc:.3f}`",
        f"- Score: `{score:.3f}`",
    ])
    display(Markdown(
        f"**Observed in this run:** best val accuracy = `{best_acc:.3f}`, final val accuracy = `{final_acc:.3f}`, "
        f"score = `{score:.3f}`. Use this together with the method card, not in isolation."
    ))
    display(Markdown(leaderboard_block))


## 4) Interactive controls

Pick a method and press **Run experiment**.  
Recommended live use: compare two or three methods under the **same regime** and discuss whether the observed behavior matches the method card.


In [ ]:
method_dd = widgets.Dropdown(
    options=list(PEFT_CARDS.keys()),
    value="lora",
    description="Method:",
    layout=widgets.Layout(width="420px"),
)

regime_dd = widgets.Dropdown(
    options=["easy", "medium", "hard"],
    value="medium",
    description="Regime:",
    layout=widgets.Layout(width="300px"),
)

train_samples_slider = widgets.IntSlider(
    value=256, min=64, max=512, step=64,
    description="Train samples:",
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
)

epochs_slider = widgets.IntSlider(
    value=15, min=5, max=40, step=5,
    description="Epochs:",
    continuous_update=False,
    layout=widgets.Layout(width="450px"),
)

lr_slider = widgets.FloatLogSlider(
    value=3e-3, base=10, min=-4, max=-1, step=0.1,
    description="LR:",
    readout_format=".1e",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)

budget_dd = widgets.Dropdown(
    options=[
        ("Tiny budget (≤ 100 params)", 100),
        ("Small budget (≤ 500 params)", 500),
        ("Medium budget (≤ 2,000 params)", 2000),
        ("No budget", None),
    ],
    value=500,
    description="Budget:",
    layout=widgets.Layout(width="420px"),
)

seed_slider = widgets.IntSlider(
    value=7, min=0, max=30, step=1,
    description="Seed:",
    continuous_update=False,
    layout=widgets.Layout(width="350px"),
)

run_btn = widgets.Button(description="Run experiment", button_style="success")
info_btn = widgets.Button(description="Show method card only")
out = widgets.Output()

def on_run(_):
    with out:
        clear_output(wait=True)
        run_and_render(
            method_dd.value,
            regime_dd.value,
            train_samples_slider.value,
            epochs_slider.value,
            lr_slider.value,
            seed_slider.value,
            budget_dd.value,
        )

def on_info(_):
    with out:
        clear_output(wait=True)
        card = PEFT_CARDS[method_dd.value]
        txt = [
            f"## {method_dd.value}",
            f"**Family:** {card['family']}",
            f"**Where it operates:** {card['where']}",
            f"**What gets updated:** {card['updates']}",
            f"**Good for:** {', '.join(card['good_for'])}",
            f"**Watch out for:** {', '.join(card['watch_out'])}",
            f"**Best when:** {card['best_when']}",
            f"**Suggested visualization:** {card['visualization']}",
        ]
        display(Markdown("\n\n".join(txt)))

run_btn.on_click(on_run)
info_btn.on_click(on_info)

controls = widgets.VBox([
    method_dd, regime_dd, budget_dd, train_samples_slider, epochs_slider, lr_slider, seed_slider,
    widgets.HBox([run_btn, info_btn]),
])

display(controls, out)


## 5) Suggested attendee prompts

Try these:

1. **Hold the regime fixed** and compare `linear_probing`, `lora`, and `prompt_tuning`.  
   Which one improves fastest? Which one ends highest?

2. Switch from **easy** to **hard**.  
   Which methods degrade the least? Which methods seem too small for the shift?

3. Keep the method fixed and vary **train samples**.  
   Does your preferred method still make sense in low-data settings?

4. Compare **LoRA vs adapters**.  
   They both change internal computation, but in different ways. Which seems more parameter-efficient here?

5. Turn this into a **budget game**.  
   Keep the budget fixed and try to maximize the score without going over the trainable-parameter limit.


## 6) How to present this in the workshop

A simple framing that works well:

- **Notebook 1:** understand *where* methods act
- **Notebook 2:** compare *how training behaves*
- **Notebook 3:** connect to a more realistic vision setup
- **This notebook:** let participants test their own hunches, play the parameter-budget game, and build a method-selection instinct

That makes this notebook the **interactive capstone**, not just another benchmark.
